# Week 9 — Thursday Lab Notebook  
## Topic: Unsupervised Learning + Extras

This lab is for **Week 9 Thursday**.  
Today we will study a different type of machine learning where the data usually does **not** have target labels for training.

We will cover:

- unsupervised learning
- clustering with K-Means
- why scaling matters
- elbow method
- silhouette score idea
- PCA intuition
- simple cluster interpretation

## 3-Hour Structure

**Hour 1**
- Understand supervised vs unsupervised learning
- Learn clustering in simple words
- Build first K-Means model
- Visualize clusters

**Hour 2**
- Understand why scaling matters
- Compare clustering before and after scaling
- Learn elbow method
- Learn silhouette score idea

**Hour 3**
- Understand PCA intuition
- Reduce data to 2 dimensions
- Visualize data using PCA
- Mini in-class practice
- After-lab tasks

## Part A — What is Unsupervised Learning?

In supervised learning, we have:

- input features
- correct answers or labels

Example:
- email features → spam or not spam
- house features → price
- student data → pass or fail

In **unsupervised learning**, we usually have:

- input features
- **no target label for training**

So the model tries to discover hidden patterns by itself.

Common unsupervised tasks are:

- clustering
- dimensionality reduction
- anomaly detection

Today our main focus is **clustering** and a little bit of **PCA**.

## Part B — Clustering in Simple Words

Clustering means:

**Put similar data points into the same group.**

Example ideas:

- group customers with similar buying behavior
- group students with similar performance patterns
- group products with similar sales patterns
- group news articles with similar content

Important point:

In clustering, the model does **not** know the correct group names in advance.  
It tries to create groups based on similarity.

## Part C — Supervised vs Unsupervised

### Supervised learning
- has labels
- learns from correct answers
- example: predict disease, predict marks, classify image

### Unsupervised learning
- no labels for training
- tries to find structure or groups
- example: customer segmentation, grouping similar records

So today, instead of predicting a target, we will **discover groups**.

## Part D — K-Means in Easy Wording

K-Means is one of the most popular clustering algorithms.

It works like this:

1. choose number of clusters `k`
2. place `k` centers
3. assign each point to the nearest center
4. update centers
5. repeat until clusters become stable

These centers are called **centroids**.

Important limitation:

K-Means needs us to choose the number of clusters in advance.

## Part E — Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

## Part F — Load a Real Dataset

We will use the **Iris dataset** from scikit-learn.

This dataset has measurements of flowers.

Feature columns:
- sepal length
- sepal width
- petal length
- petal width

For clustering practice, this dataset is very useful because:
- it is small
- it is clean
- it is easy to visualize
- we can compare cluster behavior

Note:
In real clustering, labels are usually unknown.  
But this dataset also has labels, so later we can use them only for understanding.

In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name="target")
target_names = iris.target_names

X.head()

## Part G — Inspect the Dataset

In [ ]:
print("Shape of X:", X.shape)
print("Shape of y:", y.shape)
print()
print("Feature names:")
print(list(X.columns))
print()
print("Target counts (only for understanding):")
print(y.value_counts())

In [ ]:
X.describe()

### What should students observe?

- There are **150 rows**
- There are **4 numeric features**
- Different columns have different ranges
- We can still try clustering even without using the target labels

## Part H — Start with Two Features for Easy Visualization

To make the first clustering graph easy, we will use only:

- petal length
- petal width

This makes plotting simpler for beginners.

In [ ]:
X_two = X[["petal length (cm)", "petal width (cm)"]]
X_two.head()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X_two["petal length (cm)"], X_two["petal width (cm)"])
plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.title("Raw Data (Two Features)")
plt.show()

## Part I — First K-Means Model

Let us start with `k = 3`.

Why 3?

Because the Iris dataset is known to have 3 flower classes.  
In real work, we may not know the correct number, but for first understanding this is helpful.

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_two)

cluster_labels[:20]

## Part J — Add Cluster Labels to the Data

In [ ]:
clustered_df = X_two.copy()
clustered_df["cluster"] = cluster_labels
clustered_df.head()

## Part K — Cluster Counts

In [ ]:
clustered_df["cluster"].value_counts().sort_index()

## Part L — Cluster Centers

These are the centroids found by K-Means.

In [ ]:
centers = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X_two.columns
)
centers

## Part M — Visualize the Clusters

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(clustered_df["cluster"].unique()):
    subset = clustered_df[clustered_df["cluster"] == cluster_id]
    plt.scatter(
        subset["petal length (cm)"],
        subset["petal width (cm)"],
        label=f"Cluster {cluster_id}"
    )

plt.scatter(
    centers["petal length (cm)"],
    centers["petal width (cm)"],
    marker="X",
    s=250,
    label="Centroids"
)

plt.xlabel("Petal Length")
plt.ylabel("Petal Width")
plt.title("K-Means Clusters on Iris Data")
plt.legend()
plt.show()

### What should students observe?

- Points have been divided into groups
- Each group has its own centroid
- Nearby points mostly go into the same cluster
- K-Means tries to make compact groups around the centers

## Part N — Compare with True Labels (Only for Understanding)

In real unsupervised learning, we may not have true labels.  
But here, we can compare clusters with actual flower classes just to build intuition.

In [ ]:
comparison_df = X_two.copy()
comparison_df["true_label"] = y.map({0: target_names[0], 1: target_names[1], 2: target_names[2]})
comparison_df["cluster"] = cluster_labels
comparison_df.head()

In [ ]:
pd.crosstab(comparison_df["true_label"], comparison_df["cluster"])

### Important note

Cluster numbers like `0`, `1`, `2` are just IDs.  
Cluster 0 does **not** mean class 0.

K-Means names clusters arbitrarily.

## Part O — Why Scaling Matters

K-Means depends on distance.

If one feature has a very large scale and another has a small scale, the large-scale feature may dominate the distance.

That can produce unfair clustering.

So before clustering, scaling is often very important.

## Part P — Create a Small Example Where Scale Causes Trouble

We will create a modified version of our 2-feature dataset.

We will multiply one feature by 100.  
This is only for teaching the effect of scale.

In [ ]:
X_unscaled_problem = X_two.copy()
X_unscaled_problem["petal length (cm)"] = X_unscaled_problem["petal length (cm)"] * 100

X_unscaled_problem.head()

In [ ]:
print("Original ranges:")
print(X_two.min())
print(X_two.max())
print()
print("Modified ranges:")
print(X_unscaled_problem.min())
print(X_unscaled_problem.max())

## Part Q — K-Means on the Poorly Scaled Data

In [ ]:
kmeans_bad_scale = KMeans(n_clusters=3, random_state=42, n_init=10)
bad_labels = kmeans_bad_scale.fit_predict(X_unscaled_problem)

bad_centers = pd.DataFrame(
    kmeans_bad_scale.cluster_centers_,
    columns=X_unscaled_problem.columns
)

bad_centers

In [ ]:
bad_df = X_unscaled_problem.copy()
bad_df["cluster"] = bad_labels

plt.figure(figsize=(7, 5))

for cluster_id in sorted(bad_df["cluster"].unique()):
    subset = bad_df[bad_df["cluster"] == cluster_id]
    plt.scatter(
        subset["petal length (cm)"],
        subset["petal width (cm)"],
        label=f"Cluster {cluster_id}"
    )

plt.scatter(
    bad_centers["petal length (cm)"],
    bad_centers["petal width (cm)"],
    marker="X",
    s=250,
    label="Centroids"
)

plt.xlabel("Petal Length (multiplied by 100)")
plt.ylabel("Petal Width")
plt.title("Clustering on Poorly Scaled Data")
plt.legend()
plt.show()

### What happened?

Because one feature became much larger, it strongly influenced the distance.

This is why scaling matters.

## Part R — Fix the Problem Using StandardScaler

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_unscaled_problem)

X_scaled[:5]

In [ ]:
print("Mean of scaled columns (approximately):")
print(X_scaled.mean(axis=0))
print()
print("Standard deviation of scaled columns (approximately):")
print(X_scaled.std(axis=0))

## Part S — K-Means After Scaling

In [ ]:
kmeans_scaled = KMeans(n_clusters=3, random_state=42, n_init=10)
scaled_labels = kmeans_scaled.fit_predict(X_scaled)

scaled_centers = kmeans_scaled.cluster_centers_
scaled_labels[:10]

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(np.unique(scaled_labels)):
    subset = X_scaled[scaled_labels == cluster_id]
    plt.scatter(
        subset[:, 0],
        subset[:, 1],
        label=f"Cluster {cluster_id}"
    )

plt.scatter(
    scaled_centers[:, 0],
    scaled_centers[:, 1],
    marker="X",
    s=250,
    label="Centroids"
)

plt.xlabel("Scaled Feature 1")
plt.ylabel("Scaled Feature 2")
plt.title("Clustering After Scaling")
plt.legend()
plt.show()

## Part T — Compare Cluster Counts Before and After Scaling

In [ ]:
print("Cluster counts on poorly scaled data:")
print(pd.Series(bad_labels).value_counts().sort_index())
print()
print("Cluster counts after scaling:")
print(pd.Series(scaled_labels).value_counts().sort_index())

### Main lesson

For distance-based methods like K-Means:

- scaling is often very important
- features should be on a comparable scale
- otherwise one feature may dominate the result

## Part U — How to Choose the Number of Clusters?

A common question is:

**What value of `k` should we use?**

There is no single perfect rule for all datasets.  
But two common ideas are:

- elbow method
- silhouette score

## Part V — Elbow Method in Simple Words

K-Means gives a value called **inertia**.

Inertia means, in simple words:

**How spread out the points are inside their clusters.**

Smaller inertia is generally better.  
But if we keep increasing `k`, inertia usually keeps decreasing.

So we look for a point where improvement starts becoming smaller.  
That bend is called the **elbow**.

In [ ]:
inertia_values = []

for k in range(1, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia_values.append(km.inertia_)

inertia_values

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(range(1, 9), inertia_values, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.xticks(range(1, 9))
plt.show()

### How to explain this graph in class

- As `k` increases, inertia decreases
- That is normal
- We look for the point where the drop becomes less dramatic
- That bending point is the elbow idea

Important:
Elbow method is a **guide**, not a guaranteed answer.

## Part W — Silhouette Score in Simple Words

Silhouette score checks how well the clustering is formed.

In simple words, it asks:

- are points close to their own cluster?
- are they far from other clusters?

Silhouette score usually ranges from **-1 to 1**.

Interpretation:
- closer to **1** → better-separated clusters
- around **0** → overlapping clusters
- below **0** → poor clustering

In [ ]:
silhouette_scores = []

for k in range(2, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)

silhouette_scores

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(range(2, 9), silhouette_scores, marker="o")
plt.xlabel("Number of Clusters (k)")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score for Different k Values")
plt.xticks(range(2, 9))
plt.show()

In [ ]:
score_table = pd.DataFrame({
    "k": list(range(2, 9)),
    "silhouette_score": silhouette_scores
})
score_table

### What should students observe?

- We compare different `k` values
- Higher silhouette score is generally better
- Sometimes elbow and silhouette suggest similar answers
- Sometimes they do not fully agree, so we use judgment too

## Part X — Final K-Means on All 4 Features

Until now, we used only 2 features for easy graphs.

Now let us use **all 4 Iris features**.

This is more realistic.

In [ ]:
scaler_full = StandardScaler()
X_full_scaled = scaler_full.fit_transform(X)

kmeans_full = KMeans(n_clusters=3, random_state=42, n_init=10)
full_labels = kmeans_full.fit_predict(X_full_scaled)

pd.Series(full_labels).value_counts().sort_index()

In [ ]:
pd.crosstab(y, full_labels)

## Part Y — Cluster Profiling

After clustering, we should try to understand what each cluster represents.

A simple way is to calculate the average value of each feature inside each cluster.

In [ ]:
profile_df = X.copy()
profile_df["cluster"] = full_labels

cluster_profile = profile_df.groupby("cluster").mean()
cluster_profile

### How to discuss cluster profiling

You can explain clusters like this:

- Cluster 0 has relatively larger petal measurements
- Cluster 1 has smaller petal width
- Cluster 2 has intermediate values

This is much better than saying only:
- cluster 0
- cluster 1
- cluster 2

Always try to give a **meaning** to each cluster.

## Part Z — PCA Intuition

PCA stands for **Principal Component Analysis**.

In simple words:

PCA tries to reduce many columns into fewer new columns while keeping as much useful information as possible.

Example:
- from 10 features to 2 features
- from 50 features to 3 features

Why is this useful?

- easier visualization
- less complexity
- less noise in some cases
- faster modeling sometimes

## Part AA — Important PCA Idea

PCA does **not** simply delete columns.

Instead, it creates **new features** called principal components.

These new components are combinations of the original features.

For beginners, the easiest way to think is:

**PCA compresses data into fewer dimensions while trying to keep important variation.**

## Part AB — Apply PCA to the Scaled Iris Data

In [ ]:
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_full_scaled)

X_pca[:5]

In [ ]:
print("Shape before PCA:", X_full_scaled.shape)
print("Shape after PCA:", X_pca.shape)

In [ ]:
print("Explained variance ratio:")
print(pca.explained_variance_ratio_)
print()
print("Total explained variance by 2 components:")
print(pca.explained_variance_ratio_.sum())

### What does explained variance mean?

It tells us how much information or variation is kept by each principal component.

If the total explained variance is high, then the reduced representation still keeps much of the original information.

## Part AC — Visualize the PCA Output

In [ ]:
pca_df = pd.DataFrame(X_pca, columns=["PC1", "PC2"])
pca_df["true_label"] = y

plt.figure(figsize=(7, 5))

for label in sorted(pca_df["true_label"].unique()):
    subset = pca_df[pca_df["true_label"] == label]
    plt.scatter(subset["PC1"], subset["PC2"], label=target_names[label])

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Iris Data in 2D Using PCA")
plt.legend()
plt.show()

## Part AD — K-Means on PCA Data

Now we will cluster the PCA-transformed data.

In [ ]:
kmeans_pca = KMeans(n_clusters=3, random_state=42, n_init=10)
pca_labels = kmeans_pca.fit_predict(X_pca)

pca_centers = kmeans_pca.cluster_centers_

In [ ]:
plt.figure(figsize=(7, 5))

for cluster_id in sorted(np.unique(pca_labels)):
    subset = X_pca[pca_labels == cluster_id]
    plt.scatter(subset[:, 0], subset[:, 1], label=f"Cluster {cluster_id}")

plt.scatter(
    pca_centers[:, 0],
    pca_centers[:, 1],
    marker="X",
    s=250,
    label="Centroids"
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-Means Clusters on PCA Output")
plt.legend()
plt.show()

## Part AE — Why PCA is Useful in Practice

PCA can help when:

- there are many numeric columns
- we want 2D or 3D visualization
- some columns are strongly related
- we want a compact representation

But remember:

- PCA can reduce interpretability
- principal components are not as easy to explain as original features
- scaling is usually important before PCA

## Part AF — Common Beginner Mistakes

1. Running K-Means without scaling the data  
2. Choosing `k` randomly without checking elbow or silhouette  
3. Treating cluster IDs as meaningful class names  
4. Forgetting that clustering is unsupervised  
5. Using PCA output without understanding what it does  
6. Thinking PCA keeps all information  
7. Explaining clusters without profiling them

## Part AG — Mini In-Class Practice

Do these in class:

1. Run K-Means with `k = 2` on the 2-feature Iris data  
2. Run K-Means with `k = 4` and compare cluster counts  
3. Plot inertia for `k = 1` to `k = 6`  
4. Plot silhouette scores for `k = 2` to `k = 6`  
5. Apply PCA and print explained variance ratio

## Part AH — 10 After-Lab Tasks

### Task 1
Load the Iris dataset and convert it into a Pandas DataFrame.

### Task 2
Select only two columns: petal length and petal width.

### Task 3
Fit a K-Means model with `k = 3` and print the cluster labels.

### Task 4
Create a scatter plot of the two selected features and color the points by cluster.

### Task 5
Print the centroids of the K-Means model.

### Task 6
Use `StandardScaler` on the selected features and fit K-Means again.

### Task 7
Calculate inertia values for `k = 1` to `k = 8` and draw the elbow graph.

### Task 8
Calculate silhouette scores for `k = 2` to `k = 8`.

### Task 9
Apply PCA to all 4 Iris features and reduce them to 2 components.

### Task 10
Create a cluster profile table by grouping the original dataset by cluster and taking the mean.

## Part AI — Optional Homework Challenge

Use any small CSV dataset with numeric columns.

Do the following:

1. load the dataset
2. keep useful numeric columns
3. scale the data
4. try K-Means with multiple `k` values
5. use elbow method
6. use silhouette score
7. choose one `k`
8. profile the clusters
9. apply PCA for 2D visualization
10. write 5 to 7 lines explaining what each cluster may represent

Try to explain the result in simple business or student language.

## Part AJ — Final Revision Notes

Today you learned:

- unsupervised learning means learning without target labels
- clustering groups similar points together
- K-Means is a simple and popular clustering method
- scaling is very important for distance-based methods
- elbow method helps estimate a useful `k`
- silhouette score helps judge cluster quality
- PCA reduces dimensions for easier visualization and analysis

These ideas are very common in real data science work.